In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = {
    "sp500": "^GSPC",
    "nasdaq": "^IXIC",
    "russell2000": "^RUT",
}

end = pd.Timestamp("2026-04-01")
start = end - pd.DateOffset(years=10)

window_size = 55

for name, ticker in tickers.items():
    df = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        progress=False,
    )

    prices = (
        df[["Close"]]
        .dropna()
        .loc[:end]
        .rename(columns={"Close": "Price"})
        .to_numpy()
        .flatten()
    )

    batch_size = len(prices) // window_size
    prices = prices[-batch_size * window_size:]

    price_windows = np.zeros((batch_size, window_size))

    for i in range(batch_size):
        idx = i * window_size
        window = prices[idx:idx + window_size]
        price_windows[i, :] = window / window[0]

    price_windows = pd.DataFrame(price_windows)
    price_windows.to_csv(f"{name}.csv", index=False, header=False)

    print(f"{name}.csv saved")
    print("Number of observations:", len(prices))
    print("Number of windows:", batch_size)
    print()

sp500.csv saved
Number of observations: 2475
Number of windows: 45

nasdaq.csv saved
Number of observations: 2475
Number of windows: 45

russell2000.csv saved
Number of observations: 2475
Number of windows: 45

